# Image Optimizer Notebook

Supports:
- JPEG, PNG, WebP, BMP, TIFF input
- Resize while preserving aspect ratio
- Target maximum dimensions
- Target maximum file size
- JPEG/WebP binary-search quality optimization
- PNG palette optimization


In [2]:
!pip install pillow


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from PIL import Image
from pathlib import Path
import io, os

# ==========================
# USER CONFIGURATION
# ==========================

INPUT_IMAGE = r"D:\Extension\Extension\images\test.jpg"  # Input image path
OUTPUT_FORMAT = "same"     # same, jpeg, png, webp

MAX_WIDTH = 1024
MAX_HEIGHT = 1024

TARGET_SIZE_KB = 30

RESIZE = True
COMPRESS = True
KEEP_ASPECT = True

JPEG_MIN_QUALITY = 20
JPEG_MAX_QUALITY = 95

WEBP_MIN_QUALITY = 20
WEBP_MAX_QUALITY = 95


In [4]:
def file_size_kb(path):
    return os.path.getsize(path)/1024

def image_info(path):
    img = Image.open(path)
    return {
        "format": img.format,
        "mode": img.mode,
        "width": img.width,
        "height": img.height,
        "size_kb": file_size_kb(path)
    }

def print_info(title, info):
    print("="*40)
    print(title)
    for k,v in info.items():
        print(f"{k:10}: {v}")


In [5]:
def resize_image(img):
    if not RESIZE:
        return img

    if KEEP_ASPECT:
        img.thumbnail((MAX_WIDTH, MAX_HEIGHT), Image.LANCZOS)
        return img

    return img.resize((MAX_WIDTH, MAX_HEIGHT), Image.LANCZOS)


In [6]:
def save_jpeg(img, output, target_kb):
    if img.mode in ("RGBA","LA","P"):
        img = img.convert("RGB")

    low = JPEG_MIN_QUALITY
    high = JPEG_MAX_QUALITY
    best = None

    while low <= high:
        q = (low + high)//2
        buf = io.BytesIO()

        img.save(buf,
                 format="JPEG",
                 quality=q,
                 optimize=True)

        size = len(buf.getvalue())/1024

        if size <= target_kb:
            best = buf.getvalue()
            low = q + 1
        else:
            high = q - 1

    if best is None:
        buf = io.BytesIO()
        img.save(buf,
                 format="JPEG",
                 quality=JPEG_MIN_QUALITY,
                 optimize=True)
        best = buf.getvalue()

    with open(output,"wb") as f:
        f.write(best)


In [7]:
def save_webp(img, output, target_kb):
    low = WEBP_MIN_QUALITY
    high = WEBP_MAX_QUALITY
    best = None

    while low <= high:
        q = (low+high)//2

        buf = io.BytesIO()
        img.save(buf,
                 format="WEBP",
                 quality=q,
                 method=6)

        size = len(buf.getvalue())/1024

        if size <= target_kb:
            best = buf.getvalue()
            low = q + 1
        else:
            high = q - 1

    if best is None:
        buf = io.BytesIO()
        img.save(buf,
                 format="WEBP",
                 quality=WEBP_MIN_QUALITY,
                 method=6)
        best = buf.getvalue()

    with open(output,"wb") as f:
        f.write(best)


In [8]:
def save_png(img, output, target_kb):

    # palette_sizes = [256,128,64,32,16,8,4]
    palette_sizes = [256,128,64,32]

    best = None

    for colors in palette_sizes:

        test = img.convert(
            "P",
            palette=Image.ADAPTIVE,
            colors=colors
        )

        buf = io.BytesIO()

        test.save(buf,
                  format="PNG",
                  optimize=True,
                  compress_level=9)

        size = len(buf.getvalue())/1024

        best = buf.getvalue()

        if size <= target_kb:
            print(f"Achieved target size with {colors} colors.")
            break

    with open(output,"wb") as f:
        f.write(best)


In [9]:
def optimize_image(input_path):

    info = image_info(input_path)
    print_info("Original", info)

    img = Image.open(input_path)

    img = resize_image(img)

    fmt = OUTPUT_FORMAT.lower()

    if fmt == "same":
        fmt = info["format"].lower()

    output = Path(input_path).stem + "_optimized"

    if fmt in ("jpg","jpeg"):
        output += ".jpg"
        save_jpeg(img, output, TARGET_SIZE_KB)

    elif fmt == "png":
        output += ".png"
        save_png(img, output, TARGET_SIZE_KB)

    elif fmt == "webp":
        output += ".webp"
        save_webp(img, output, TARGET_SIZE_KB)

    else:
        raise ValueError(f"Unsupported output format: {fmt}")

    print()
    print_info("Optimized", image_info(output))

    reduction = (
        100
        * (1-file_size_kb(output)/info["size_kb"])
    )

    print(f"Reduction: {reduction:.2f}%")

    return output


In [10]:
# output = optimize_image(INPUT_IMAGE)
# print("Saved:", output)

# from IPython.display import display

# display(Image.open(INPUT_IMAGE))
# display(Image.open(output))


In [11]:
import sys

!{sys.executable} -m pip install "rembg[cpu]"


[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from PIL import Image
from rembg import remove

print("rembg import OK")

# 1. Load image and strip background to get transparency
input_img = Image.open(INPUT_IMAGE)
isolated_img = remove(input_img)  # RGBA image with transparent background

# 2. Create white background
white_bg = Image.new("RGBA", isolated_img.size, (255, 255, 255, 255))

# 3. Composite
final_img = Image.alpha_composite(white_bg, isolated_img).convert("RGB")

# 4. Save output
output_path = r"D:\Extension\Extension\testing_img_compression\white_background.png"
final_img.save(output_path)
print(f"Saved: {output_path}")

rembg import OK


100%|########################################| 176M/176M [00:00<00:00, 177GB/s]


Saved: D:\Extension\Extension\testing_img_compression\white_background.png


In [14]:
from PIL import Image

# Open the image
img = Image.open(r"D:\Extension\Extension\testing_img_compression\white_background.png").convert("RGB")

# Get color of the top-left corner pixel (x=0, y=0)
r, g, b = img.getpixel((0, 0))

# Convert RGB to HEX format
hex_color = f"#{r:02x}{g:02x}{b:02x}"

print(f"RGB Color: ({r}, {g}, {b})")
print(f"HEX Color: {hex_color.upper()}")

# Quick check if it's pure white
if hex_color.upper() == "#FFFFFF":
    print("The background is pure white.")
else:
    print("The background is NOT pure white.")

RGB Color: (255, 255, 255)
HEX Color: #FFFFFF
The background is pure white.
